<a href="https://colab.research.google.com/github/muthuprakash2006a-collab/NM-syntax/blob/main/Syntax_surgeon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install -q gradio transformers torch autopep8 black reportlab

In [10]:
import gradio as gr
import torch
import ast
import re
from transformers import pipeline
import autopep8
import black
from datetime import datetime
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

In [11]:
MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"

device = 0 if torch.cuda.is_available() else -1

print("Loading model...")

pipe = pipeline(
    "text-generation",
    model=MODEL_NAME,
    device=device
)

print("Model loaded successfully")

Loading model...


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

Model loaded successfully


In [12]:
def check_syntax(code):
    try:
        ast.parse(code)
        return True, "No syntax errors"
    except SyntaxError as e:
        return False, f"Syntax Error line {e.lineno}: {e.msg}"

In [13]:
def format_code(code):

    try:
        return black.format_str(code, mode=black.FileMode())

    except Exception:

        return autopep8.fix_code(code)

In [14]:
def detect_python_code(text):

    patterns = [
        r"def\s+\w+",
        r"class\s+\w+",
        r"import\s+\w+",
        r"for\s+.*\s+in",
        r"while\s+.*:"
    ]

    for p in patterns:
        if re.search(p, text):
            return True

    return False

In [15]:
def chatbot(user_input):

    if detect_python_code(user_input):

        valid, message = check_syntax(user_input)

        fixed_code = format_code(user_input)

        return f"{message}\n\nFormatted Code:\n\n{fixed_code}"

    else:

        response = pipe(
            user_input,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7
        )

        return response[0]["generated_text"]

In [16]:
def respond(message, history):

    reply = chatbot(message)

    history.append((message, reply))

    return "", history


with gr.Blocks() as demo:

    gr.Markdown("# Granite AI Chatbot + Python Fixer")

    chatbot_ui = gr.Chatbot()

    msg = gr.Textbox()

    clear = gr.Button("Clear")

    msg.submit(respond, [msg, chatbot_ui], [msg, chatbot_ui])

    clear.click(lambda: None, None, chatbot_ui, queue=False)


demo.launch(share=True)

/tmp/ipykernel_161/1124378156.py:14: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_ui = gr.Chatbot()
/tmp/ipykernel_161/1124378156.py:14: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot_ui = gr.Chatbot()


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6c8828a56f79dbc404.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
